# Notebook 151: Word2Vec と静的埋め込み

## Word2Vec & Static Embeddings

---

### このノートブックの位置づけ

**難易度: ★★★☆☆ | 所要時間: 120-150 分 | カテゴリ: Embeddings**

前回の Notebook 150（埋め込みの幾何学）では、ベクトル空間における単語表現の基本的な性質を学びました。本ノートブックでは、そのようなベクトル表現を **データから自動的に学習する** 手法、特に **Word2Vec** を中心に、静的埋め込みの代表的手法を体系的に学びます。

### 学習目標

- [ ] Skip-gramモデルの仕組みを理論から理解し、NumPyで実装できる
- [ ] 負例サンプリング（Negative Sampling）がなぜ必要かを説明できる
- [ ] CBOWとSkip-gramの違いを比較できる
- [ ] FastText（サブワード）とGloVeの仕組みを理解し、Word2Vecと比較できる
- [ ] 学習過程を可視化し、埋め込みの品質を評価できる

### 前提知識

- ✅ Notebook 150（埋め込みの幾何学）
- ✅ Notebook 70-76（ニューラルエンジン、勾配降下法）
- ✅ NumPyの基礎

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [なぜ「学習する」埋め込みが必要か？](#2-なぜ学習する埋め込みが必要か)
3. [Skip-gram モデルの数理](#3-skip-gram-モデルの数理)
4. [Skip-gram を NumPy でスクラッチ実装](#4-skip-gram-を-numpy-でスクラッチ実装)
5. [学習結果の可視化](#5-学習結果の可視化)
6. [CBOW vs Skip-gram の比較](#6-cbow-vs-skip-gram-の比較)
7. [GloVe — 共起行列の因数分解](#7-glove--共起行列の因数分解)
8. [FastText — サブワードの力](#8-fasttext--サブワードの力)
9. [静的埋め込みの限界](#9-静的埋め込みの限界)
10. [まとめ・チートシート・よくある間違い・自己評価クイズ](#10-まとめチートシートよくある間違い自己評価クイズ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# セクション1: 環境セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定（環境に応じて調整）
plt.rcParams['font.family'] = ['Hiragino Sans', 'Arial Unicode MS', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Seaborn スタイル
sns.set_style('whitegrid')

# 再現性のためのシード
np.random.seed(42)

print("✅ 環境セットアップ完了")
print(f"NumPy version: {np.__version__}")

---

## 2. なぜ「学習する」埋め込みが必要か？

### 2.1 手作り特徴量の限界

従来のNLPでは、単語をone-hotベクトルや手作りの特徴量（品詞、長さ、出現頻度など）で表現していました。しかし、この方法には重大な問題があります：

| 問題 | 説明 |
|------|------|
| **高次元** | 語彙サイズ V に比例した次元数（V = 50,000 なら 50,000次元） |
| **スパース** | ほとんどの要素が 0 |
| **意味を捉えない** | one-hot では "king" と "queen" の距離 = "king" と "apple" の距離 |
| **スケールしない** | 新しい単語を追加するたびに全次元が変わる |

### 2.2 分布仮説 (Distributional Hypothesis)

Word2Vec の根底にある哲学は、J.R. Firth (1957) の有名な言葉に集約されます：

> **"You shall know a word by the company it keeps."**
> （「単語はその周囲の単語によって特徴づけられる」）

つまり、**似た文脈に出現する単語は、似た意味を持つ** という仮説です。

例えば：
- "犬が公園で**走る**" と "猫が庭で**走る**" → "犬" と "猫" は似た文脈に出現
- "コーヒーを**飲む**" と "紅茶を**飲む**" → "コーヒー" と "紅茶" は似た文脈に出現

### 2.3 コンテキストウィンドウ

文脈を捉えるために、中心語の周囲にウィンドウをスライドさせます：

```
文: "犬  が  公園  で  走る"

window_size = 2 の場合:

     [犬  が  公園  で  走る]
          ↑    ★    ↑
        文脈  中心語  文脈

中心語="公園" → 文脈=["犬", "が", "で", "走る"]

     [犬  が  公園  で  走る]
      ↑   ★    ↑
    文脈 中心語  文脈

中心語="が" → 文脈=["犬", "公園"]
```

### 2.4 Skip-gram vs CBOW

Word2Vec には2つのアーキテクチャがあります：

```
Skip-gram:  中心語 → 文脈語を予測
             "公園" → ["犬", "が", "で", "走る"]

CBOW:       文脈語 → 中心語を予測
             ["犬", "が", "で", "走る"] → "公園"
```

本ノートブックでは、まず **Skip-gram** を詳しく実装し、その後 CBOW と比較します。

---

## 3. Skip-gram モデルの数理

### 3.1 基本的な定式化

Skip-gram モデルの目的は、中心語 $w$ が与えられたときに、文脈語 $c$ が出現する確率を最大化することです：

$$
P(c \mid w) = \frac{\exp(\mathbf{v}_c' \cdot \mathbf{v}_w)}{\sum_{j=1}^{V} \exp(\mathbf{v}_j' \cdot \mathbf{v}_w)}
$$

ここで：
- $\mathbf{v}_w \in \mathbb{R}^D$: 中心語 $w$ の入力側埋め込みベクトル（行列 $W_{\text{in}}$ の行）
- $\mathbf{v}_c' \in \mathbb{R}^D$: 文脈語 $c$ の出力側埋め込みベクトル（行列 $W_{\text{out}}$ の行）
- $V$: 語彙サイズ
- $D$: 埋め込み次元数

### 3.2 損失関数

コーパス全体にわたる対数尤度を最大化します：

$$
\mathcal{L} = \sum_{(w, c) \in \mathcal{D}} \log P(c \mid w)
$$

### 3.3 問題：ソフトマックスの計算コスト

上記の確率計算には、**語彙全体** にわたる和（分母の $\sum_{j=1}^{V}$）が必要です。語彙サイズが $V = 100,000$ のとき、毎回10万個のベクトルとの内積を計算する必要があり、非常に高コストです。

### 3.4 解決策：負例サンプリング (Negative Sampling)

語彙全体のソフトマックスの代わりに、**少数の負例（ランダムに選んだ「文脈にない」単語）** だけを使って学習します。

#### 目的関数

正例ペア $(w, c)$ と $K$ 個の負例 $n_1, n_2, \ldots, n_K$ に対して：

$$
\mathcal{L} = \log \sigma(\mathbf{v}_c' \cdot \mathbf{v}_w) + \sum_{k=1}^{K} \log \sigma(-\mathbf{v}_{n_k}' \cdot \mathbf{v}_w)
$$

ここで $\sigma(x) = \frac{1}{1 + e^{-x}}$ はシグモイド関数です。

**直感的な理解：**
- 第1項：正例の内積を **大きくする**（$\sigma$ が 1 に近づく）
- 第2項：負例の内積を **小さくする**（$\sigma(-x)$ が 1 に近づく）

#### 負例のサンプリング分布

負例は頻度に基づいてサンプリングしますが、3/4乗で補正します：

$$
P_{\text{neg}}(w) \propto \text{freq}(w)^{3/4}
$$

この 3/4 乗は、低頻度語のサンプリング確率を相対的に上げ、高頻度語の支配を緩和します。

### 3.5 Skip-gram アーキテクチャ図

```
入力（one-hot）    入力埋め込み      出力埋め込み     Sigmoid
                    W_in (V×D)       W_out (V×D)

  [0]               ┌───┐
  [0]               │   │             ┌───┐
  [1] ─── Lookup ──→│v_w│── dot ──→  │v_c│ ──→ σ(v_c · v_w) ≈ 1  (正例)
  [0]               │   │             └───┘
  [0]               │   │             ┌───┐
   ⋮                │   │── dot ──→  │v_n│ ──→ σ(-v_n · v_w) ≈ 1  (負例)
                    └───┘             └───┘

  中心語 w                           文脈語 c / 負例 n
```

**重要なポイント：**
- 2つの別々の埋め込み行列 $W_{\text{in}}$ と $W_{\text{out}}$ を使う
- 最終的に使うのは $W_{\text{in}}$（入力側の埋め込み）
- V 次元のソフトマックスが不要 → 高速に学習できる

---

## 4. Skip-gram を NumPy でスクラッチ実装

ここから、Skip-gram with Negative Sampling を NumPy だけで一から実装します。

In [ ]:
# ============================================================
# セクション4.1: トイコーパスの準備
# ============================================================

# 小さなコーパス（動物・食べ物・自然に関する文）
# 意味的なクラスタが形成されるように設計
corpus_sentences = [
    # 動物グループ
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the mouse",
    "the dog chased the cat",
    "a cat and a dog are pets",
    "the mouse ran from the cat",
    "the dog ran in the park",
    # 食べ物グループ
    "i eat rice and fish",
    "i drink water and juice",
    "she eats bread and cheese",
    "he drinks coffee and tea",
    "rice and bread are food",
    "fish and cheese are tasty",
    # 自然グループ
    "the sun shines in the sky",
    "the moon glows in the sky",
    "rain falls from the sky",
    "the river flows to the sea",
    "the sun and moon are bright",
    # 追加の文脈を提供する文
    "the cat sleeps on the mat",
    "the dog sleeps on the rug",
    "i eat fish and rice for dinner",
    "she drinks tea and coffee",
    "the sun rises in the east",
    "the river meets the sea",
]

# トークナイズ（簡易：スペース分割）
corpus_tokenized = [sentence.split() for sentence in corpus_sentences]

# 全トークンをフラットなリストにする
all_tokens = [token for sentence in corpus_tokenized for token in sentence]

print(f"✅ コーパス統計:")
print(f"   文の数:     {len(corpus_sentences)}")
print(f"   総トークン数: {len(all_tokens)}")
print(f"   サンプル文:  '{corpus_sentences[0]}'")

In [ ]:
# ============================================================
# セクション4.2: 語彙の構築
# ============================================================

# 単語の出現頻度をカウント
word_counts = Counter(all_tokens)

# 語彙の構築（全単語を含める。実用では低頻度語をフィルタする）
vocab = sorted(word_counts.keys())
vocab_size = len(vocab)

# 単語 → インデックス / インデックス → 単語 の辞書
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

# 頻度順に表示
print(f"✅ 語彙サイズ: {vocab_size}")
print(f"\n📊 上位15語の出現頻度:")
for word, count in word_counts.most_common(15):
    idx = word2idx[word]
    print(f"   [{idx:>3d}] '{word}': {count}回")

In [ ]:
# ============================================================
# セクション4.3: 学習ペアの生成（中心語, 文脈語）
# ============================================================

def generate_training_pairs(tokenized_corpus, word2idx, window_size=2):
    """
    Skip-gram の学習ペアを生成する。
    各文について、中心語とウィンドウ内の文脈語のペア (center_idx, context_idx) を作る。
    
    Parameters:
        tokenized_corpus: list of list of str - トークナイズ済みの文のリスト
        word2idx: dict - 単語→インデックスの辞書
        window_size: int - 中心語からの最大距離
    
    Returns:
        training_pairs: list of (int, int) - (中心語idx, 文脈語idx) のリスト
    """
    training_pairs = []
    
    for sentence in tokenized_corpus:
        indices = [word2idx[w] for w in sentence]
        
        for center_pos in range(len(indices)):
            center_idx = indices[center_pos]
            
            # ウィンドウ内の文脈語をイテレート
            for offset in range(-window_size, window_size + 1):
                if offset == 0:  # 中心語自身はスキップ
                    continue
                
                context_pos = center_pos + offset
                
                # 文の範囲外チェック
                if 0 <= context_pos < len(indices):
                    context_idx = indices[context_pos]
                    training_pairs.append((center_idx, context_idx))
    
    return training_pairs


# ウィンドウサイズ2で学習ペアを生成
WINDOW_SIZE = 2
training_pairs = generate_training_pairs(corpus_tokenized, word2idx, WINDOW_SIZE)

print(f"✅ 学習ペア生成完了")
print(f"   ウィンドウサイズ: {WINDOW_SIZE}")
print(f"   ペア総数: {len(training_pairs)}")
print(f"\n📋 最初の10ペア:")
for center, context in training_pairs[:10]:
    print(f"   中心語='{idx2word[center]}' → 文脈語='{idx2word[context]}'")

In [ ]:
# ============================================================
# セクション4.4: 負例サンプリングの実装
# ============================================================

def build_negative_sampling_distribution(word_counts, vocab, power=0.75):
    """
    負例サンプリング用の確率分布を構築する。
    P(w) ∝ freq(w)^0.75
    
    Parameters:
        word_counts: Counter - 単語の出現頻度
        vocab: list - 語彙リスト
        power: float - 頻度の指数（デフォルト: 0.75）
    
    Returns:
        neg_probs: np.ndarray - 各単語のサンプリング確率
    """
    # 各単語の頻度の0.75乗を計算
    freqs = np.array([word_counts[w] for w in vocab], dtype=np.float64)
    powered = freqs ** power
    # 正規化して確率分布にする
    neg_probs = powered / powered.sum()
    return neg_probs


# 負例サンプリング分布を構築
neg_probs = build_negative_sampling_distribution(word_counts, vocab)

# 元の頻度分布と比較
raw_freqs = np.array([word_counts[w] for w in vocab], dtype=np.float64)
raw_probs = raw_freqs / raw_freqs.sum()

# 上位10語で比較表示
print("📊 サンプリング確率の比較（上位10語）:")
print(f"{'単語':>10s}  {'元の確率':>10s}  {'0.75乗補正後':>12s}  {'変化':>8s}")
print("-" * 48)

top_words = [w for w, _ in word_counts.most_common(10)]
for w in top_words:
    idx = word2idx[w]
    ratio = neg_probs[idx] / raw_probs[idx]
    direction = "↓" if ratio < 1 else "↑"
    print(f"'{w:>8s}'  {raw_probs[idx]:>10.4f}  {neg_probs[idx]:>12.4f}  {direction} {ratio:.2f}x")

print("\n💡 高頻度語の確率は下がり、低頻度語の確率が相対的に上がります")

In [ ]:
# ============================================================
# セクション4.5: Skip-gram モデルの実装
# ============================================================

class SkipGramNS:
    """
    Skip-gram with Negative Sampling の NumPy 実装。
    
    2つの埋め込み行列を学習する：
    - W_in  (V × D): 入力（中心語）側の埋め込み
    - W_out (V × D): 出力（文脈語）側の埋め込み
    
    最終的に使用するのは W_in。
    """
    
    def __init__(self, vocab_size, embedding_dim, neg_probs, num_negatives=5):
        """
        Parameters:
            vocab_size: int - 語彙サイズ V
            embedding_dim: int - 埋め込み次元 D
            neg_probs: np.ndarray - 負例サンプリング確率
            num_negatives: int - 負例の数 K
        """
        self.V = vocab_size
        self.D = embedding_dim
        self.neg_probs = neg_probs
        self.K = num_negatives
        
        # 埋め込み行列の初期化（小さなランダム値）
        # Xavier初期化に近い: std = 1/sqrt(D)
        scale = 1.0 / np.sqrt(embedding_dim)
        self.W_in = np.random.randn(vocab_size, embedding_dim) * scale
        self.W_out = np.random.randn(vocab_size, embedding_dim) * scale
    
    @staticmethod
    def sigmoid(x):
        """数値安定なシグモイド関数"""
        # オーバーフロー対策: clipping
        x = np.clip(x, -15, 15)
        return 1.0 / (1.0 + np.exp(-x))
    
    def sample_negatives(self, positive_idx):
        """
        負例をサンプリングする。
        正例のインデックスと同じものは避ける。
        
        Parameters:
            positive_idx: int - 正例（文脈語）のインデックス
        
        Returns:
            neg_indices: np.ndarray - 負例のインデックス配列
        """
        neg_indices = []
        while len(neg_indices) < self.K:
            candidates = np.random.choice(
                self.V, size=self.K - len(neg_indices), p=self.neg_probs
            )
            # 正例と同じものを除外
            for c in candidates:
                if c != positive_idx:
                    neg_indices.append(c)
                    if len(neg_indices) >= self.K:
                        break
        return np.array(neg_indices)
    
    def train_step(self, center_idx, context_idx, learning_rate):
        """
        1つの学習ペアに対する forward → loss → backward → update。
        
        Parameters:
            center_idx: int - 中心語のインデックス
            context_idx: int - 文脈語（正例）のインデックス
            learning_rate: float - 学習率
        
        Returns:
            loss: float - この学習ペアの損失
        """
        # --- Forward Pass ---
        # 中心語の埋め込みベクトル (D,)
        v_center = self.W_in[center_idx]  
        
        # 正例: 文脈語の出力埋め込み (D,)
        v_context = self.W_out[context_idx]
        
        # 正例のスコアと損失
        pos_score = np.dot(v_context, v_center)     # スカラー
        pos_sigmoid = self.sigmoid(pos_score)       # σ(v_c · v_w)
        pos_loss = -np.log(pos_sigmoid + 1e-10)     # -log σ(v_c · v_w)
        
        # 負例をサンプリング
        neg_indices = self.sample_negatives(context_idx)
        
        # 負例のスコアと損失
        v_negatives = self.W_out[neg_indices]        # (K, D)
        neg_scores = v_negatives @ v_center          # (K,)
        neg_sigmoids = self.sigmoid(-neg_scores)     # σ(-v_n · v_w)
        neg_loss = -np.sum(np.log(neg_sigmoids + 1e-10))  # -Σ log σ(-v_n · v_w)
        
        # 合計損失
        loss = pos_loss + neg_loss
        
        # --- Backward Pass ---
        # 勾配の計算
        # ∂L/∂v_center（中心語の勾配）
        # 正例からの勾配: (σ(s) - 1) * v_context
        grad_center = (pos_sigmoid - 1.0) * v_context
        
        # 負例からの勾配: Σ (1 - σ(-s_n)) * v_n = Σ σ(s_n) * v_n
        neg_sigmoid_pos = self.sigmoid(neg_scores)  # σ(v_n · v_w)
        grad_center += neg_sigmoid_pos @ v_negatives  # (K,) @ (K, D) → (D,)
        
        # ∂L/∂v_context（正例の文脈語の勾配）: (σ(s) - 1) * v_center
        grad_context = (pos_sigmoid - 1.0) * v_center
        
        # ∂L/∂v_neg（各負例の勾配）: σ(s_n) * v_center
        grad_negatives = np.outer(neg_sigmoid_pos, v_center)  # (K, D)
        
        # --- SGD Update ---
        self.W_in[center_idx] -= learning_rate * grad_center
        self.W_out[context_idx] -= learning_rate * grad_context
        self.W_out[neg_indices] -= learning_rate * grad_negatives
        
        return loss
    
    def get_embedding(self, word_idx):
        """単語の埋め込みベクトルを取得（入力側）"""
        return self.W_in[word_idx]
    
    def get_all_embeddings(self):
        """全単語の埋め込み行列を取得（入力側）"""
        return self.W_in.copy()


print("✅ SkipGramNS クラスの定義完了")
print(f"   メソッド: __init__, sigmoid, sample_negatives, train_step, get_embedding")

In [ ]:
# ============================================================
# セクション4.6: 学習ループの実行
# ============================================================

# ハイパーパラメータ
EMBEDDING_DIM = 30       # 埋め込み次元（可視化のため小さめ）
NUM_NEGATIVES = 5        # 負例の数
LEARNING_RATE = 0.025    # 初期学習率
NUM_EPOCHS = 800         # エポック数
LR_DECAY = True          # 学習率の線形減衰

# モデルの初期化
np.random.seed(42)
model = SkipGramNS(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    neg_probs=neg_probs,
    num_negatives=NUM_NEGATIVES
)

print(f"📋 ハイパーパラメータ:")
print(f"   語彙サイズ V = {vocab_size}")
print(f"   埋め込み次元 D = {EMBEDDING_DIM}")
print(f"   負例数 K = {NUM_NEGATIVES}")
print(f"   学習率 η = {LEARNING_RATE}")
print(f"   エポック数 = {NUM_EPOCHS}")
print(f"   W_in shape = {model.W_in.shape}")
print(f"   W_out shape = {model.W_out.shape}")
print(f"   学習ペア数/エポック = {len(training_pairs)}")
print(f"\n🚀 学習開始...")

# 学習ループ
loss_history = []

for epoch in range(NUM_EPOCHS):
    # エポックごとにデータをシャッフル
    np.random.shuffle(training_pairs)
    
    # 学習率の線形減衰
    if LR_DECAY:
        lr = LEARNING_RATE * (1.0 - epoch / NUM_EPOCHS)
        lr = max(lr, LEARNING_RATE * 0.01)  # 最小学習率
    else:
        lr = LEARNING_RATE
    
    epoch_loss = 0.0
    
    for center_idx, context_idx in training_pairs:
        loss = model.train_step(center_idx, context_idx, lr)
        epoch_loss += loss
    
    avg_loss = epoch_loss / len(training_pairs)
    loss_history.append(avg_loss)
    
    # 進捗表示（100エポックごと）
    if (epoch + 1) % 100 == 0:
        print(f"   Epoch {epoch+1:>4d}/{NUM_EPOCHS} | 平均損失: {avg_loss:.4f} | 学習率: {lr:.6f}")

print(f"\n✅ 学習完了! 最終損失: {loss_history[-1]:.4f}")

In [ ]:
# ============================================================
# セクション4.7: 学習損失曲線のプロット
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左: 全体の損失曲線
axes[0].plot(loss_history, linewidth=1.5, color='steelblue', alpha=0.8)
axes[0].set_xlabel('エポック', fontsize=12)
axes[0].set_ylabel('平均損失', fontsize=12)
axes[0].set_title('Skip-gram 学習損失曲線', fontsize=14)
axes[0].grid(True, alpha=0.3)

# 右: 後半の損失曲線（収束の詳細）
start_idx = len(loss_history) // 4
axes[1].plot(
    range(start_idx, len(loss_history)),
    loss_history[start_idx:],
    linewidth=1.5, color='coral', alpha=0.8
)
axes[1].set_xlabel('エポック', fontsize=12)
axes[1].set_ylabel('平均損失', fontsize=12)
axes[1].set_title(f'エポック {start_idx}～ の損失推移（拡大）', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"📊 損失の変化: {loss_history[0]:.4f} → {loss_history[-1]:.4f}")
print(f"   削減率: {(1 - loss_history[-1]/loss_history[0])*100:.1f}%")

---

## 5. 学習結果の可視化

学習された 30 次元の埋め込みを PCA で 2 次元に圧縮し、単語間の関係を可視化します。

In [ ]:
# ============================================================
# セクション5.1: PCA による2次元可視化
# ============================================================

# 埋め込みの取得
embeddings = model.get_all_embeddings()  # (V, D)

# PCA で2次元に削減
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

print(f"✅ PCA 寄与率: PC1={pca.explained_variance_ratio_[0]:.3f}, "
      f"PC2={pca.explained_variance_ratio_[1]:.3f}")
print(f"   合計: {sum(pca.explained_variance_ratio_[:2]):.3f}")

# カテゴリで色分け
animal_words = {'cat', 'dog', 'mouse', 'pets', 'chased', 'ran'}
food_words = {'eat', 'eats', 'drink', 'drinks', 'rice', 'fish',
              'bread', 'cheese', 'water', 'juice', 'coffee', 'tea', 'food', 'tasty', 'dinner'}
nature_words = {'sun', 'moon', 'sky', 'rain', 'river', 'sea',
                'shines', 'glows', 'falls', 'flows', 'rises', 'bright', 'east', 'meets'}

fig, ax = plt.subplots(figsize=(12, 9))

for i, word in enumerate(vocab):
    x, y = embeddings_2d[i]
    
    # カテゴリに基づいて色を決定
    if word in animal_words:
        color = 'red'
        marker = 'o'
    elif word in food_words:
        color = 'blue'
        marker = 's'
    elif word in nature_words:
        color = 'green'
        marker = '^'
    else:
        color = 'gray'
        marker = '.'
    
    ax.scatter(x, y, c=color, marker=marker, s=80, alpha=0.7, edgecolors='white', linewidths=0.5)
    ax.annotate(word, (x, y), fontsize=9, ha='center', va='bottom',
                xytext=(0, 5), textcoords='offset points')

# 凡例
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
           markersize=10, label='動物関連'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='blue',
           markersize=10, label='食べ物関連'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='green',
           markersize=10, label='自然関連'),
    Line2D([0], [0], marker='.', color='w', markerfacecolor='gray',
           markersize=10, label='機能語等'),
]
ax.legend(handles=legend_elements, loc='best', fontsize=11)

ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_title('Skip-gram 学習済み埋め込みの2D可視化（PCA）', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 意味的に関連する単語がクラスタを形成していることを確認してください")

In [ ]:
# ============================================================
# セクション5.2: 最近傍探索
# ============================================================

def find_nearest_neighbors(word, model, word2idx, idx2word, top_k=5):
    """
    コサイン類似度に基づいて最近傍の単語を検索する。
    
    Parameters:
        word: str - クエリ単語
        model: SkipGramNS - 学習済みモデル
        word2idx: dict - 単語→インデックス辞書
        idx2word: dict - インデックス→単語辞書
        top_k: int - 返す近傍数
    
    Returns:
        list of (str, float) - (単語, 類似度) のリスト
    """
    if word not in word2idx:
        print(f"⚠️ '{word}' は語彙にありません")
        return []
    
    word_idx = word2idx[word]
    query_vec = model.get_embedding(word_idx)
    
    # 全埋め込みとのコサイン類似度を計算
    all_embeddings = model.get_all_embeddings()  # (V, D)
    
    # ノルムの計算
    query_norm = np.linalg.norm(query_vec)
    all_norms = np.linalg.norm(all_embeddings, axis=1)
    
    # コサイン類似度
    cosine_sims = (all_embeddings @ query_vec) / (all_norms * query_norm + 1e-10)
    
    # 自分自身を除外してソート
    cosine_sims[word_idx] = -np.inf
    top_indices = np.argsort(cosine_sims)[::-1][:top_k]
    
    results = [(idx2word[idx], cosine_sims[idx]) for idx in top_indices]
    return results


# いくつかの単語で最近傍を検索
query_words = ['cat', 'rice', 'sun', 'coffee', 'river']

print("📊 学習済み埋め込みでの最近傍探索:")
print("=" * 50)

for query in query_words:
    neighbors = find_nearest_neighbors(query, model, word2idx, idx2word, top_k=5)
    print(f"\n🔍 '{query}' の最近傍:")
    for rank, (word, sim) in enumerate(neighbors, 1):
        bar = '█' * int(max(0, sim) * 20)
        print(f"   {rank}. '{word}' (類似度: {sim:.4f}) {bar}")

In [ ]:
# ============================================================
# セクション5.3: 類似度ヒートマップ
# ============================================================

# 注目する単語を選択
focus_words = ['cat', 'dog', 'mouse', 'rice', 'fish', 'bread',
               'sun', 'moon', 'sky', 'river', 'sea']
focus_indices = [word2idx[w] for w in focus_words]
focus_embeddings = embeddings[focus_indices]

# コサイン類似度行列の計算
norms = np.linalg.norm(focus_embeddings, axis=1, keepdims=True)
normalized = focus_embeddings / (norms + 1e-10)
similarity_matrix = normalized @ normalized.T

# ヒートマップ
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    similarity_matrix,
    xticklabels=focus_words,
    yticklabels=focus_words,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax
)
ax.set_title('学習済み埋め込みのコサイン類似度ヒートマップ', fontsize=14)

plt.tight_layout()
plt.show()

print("💡 同じカテゴリの単語間で高い類似度（赤色）が見られます")
print("   異なるカテゴリ間では低い類似度（青色）が見られます")

---

## 6. CBOW vs Skip-gram の比較

### 6.1 CBOW (Continuous Bag of Words) の仕組み

CBOW は Skip-gram の「逆」です：

```
Skip-gram: 1つの中心語 → 複数の文脈語を予測
   入力: "公園"  → 出力: ["犬", "が", "で", "走る"]

CBOW:     複数の文脈語 → 1つの中心語を予測
   入力: ["犬", "が", "で", "走る"]  → 出力: "公園"
```

CBOW のアーキテクチャ：

```
 文脈語1 ───→ v_1 ─┐
 文脈語2 ───→ v_2 ─┼→ 平均 ─→ v_avg ─→ softmax/NS → 中心語
 文脈語3 ───→ v_3 ─┤
 文脈語4 ───→ v_4 ─┘
```

文脈語の埋め込みを**平均**して、中心語を予測します。

### 6.2 比較表

| 観点 | Skip-gram | CBOW |
|------|-----------|------|
| **入力** | 中心語 1語 | 文脈語 2×window語 |
| **出力** | 文脈語を予測 | 中心語を予測 |
| **学習ペア数** | 多い（各中心語から複数ペア） | 少ない（ウィンドウごとに1ペア） |
| **低頻度語** | ✅ 得意（個別に学習される） | ⚠️ 苦手（文脈に埋もれる） |
| **高頻度語** | 普通 | ✅ 得意（文脈の平均化が安定） |
| **学習速度** | やや遅い | 速い |
| **小さなデータセット** | ✅ 有利 | ⚠️ 不利 |
| **大きなデータセット** | 普通 | ✅ 有利 |
| **原論文** | Mikolov et al. (2013a) | Mikolov et al. (2013a) |

### 6.3 いつどちらを使うか？

- **Skip-gram** を選ぶ場面：
  - データセットが小さい
  - 低頻度語の品質が重要
  - 単語のニュアンスの違いを捉えたい

- **CBOW** を選ぶ場面：
  - データセットが大きい
  - 学習速度が重要
  - 高頻度語の品質が重要

実用的には、**gensim** や **fasttext** ライブラリがどちらもサポートしており、データに合わせて選択できます。

---

## 7. GloVe — 共起行列の因数分解

### 7.1 GloVe の基本的なアイデア

**GloVe** (Global Vectors for Word Representation, Pennington et al., 2014) は、Word2Vec とは異なるアプローチを取ります：

| | Word2Vec | GloVe |
|---|---------|-------|
| **手法** | ローカルな文脈ウィンドウで予測 | グローバルな共起統計量を活用 |
| **情報源** | 局所的な (center, context) ペア | コーパス全体の共起行列 |
| **学習方法** | SGD（ペアごとに更新） | 重み付き最小二乗法 |

GloVe の核心的なアイデアは、**共起行列の対数をベクトルの内積で近似する** ことです：

$$
\mathbf{v}_i \cdot \mathbf{v}_j + b_i + b_j \approx \log X_{ij}
$$

ここで $X_{ij}$ は単語 $i$ と単語 $j$ の共起回数です。

### 7.2 GloVe の損失関数

$$
J = \sum_{i,j=1}^{V} f(X_{ij}) \left( \mathbf{v}_i \cdot \mathbf{v}_j + b_i + b_j - \log X_{ij} \right)^2
$$

重み関数 $f(x)$ は:
- 共起回数が 0 のペアを無視
- 非常に高頻度のペアの影響を抑制

$$
f(x) = \begin{cases}
(x / x_{\max})^\alpha & \text{if } x < x_{\max} \\
1 & \text{otherwise}
\end{cases}
$$

($\alpha = 0.75$, $x_{\max} = 100$ が典型的)

### 7.3 なぜ共起行列の分解が意味的関係を捉えるのか？

GloVe 論文の重要な洞察は、**共起確率の比** が意味的関係を表すことです：

| | P(k \| ice) | P(k \| steam) | 比率 |
|---|-----------|-------------|------|
| k = solid | 高い | 低い | **大きい** |
| k = gas | 低い | 高い | **小さい** |
| k = water | 高い | 高い | ≈ 1 |
| k = fashion | 低い | 低い | ≈ 1 |

In [ ]:
# ============================================================
# セクション7.4: 共起行列の構築と可視化
# ============================================================

def build_cooccurrence_matrix(tokenized_corpus, word2idx, window_size=2):
    """
    共起行列を構築する。
    X[i][j] = 単語 i と単語 j がウィンドウ内で共起した回数。
    距離による重み付け（近い単語ほど重み大）も適用する。
    
    Parameters:
        tokenized_corpus: list of list of str
        word2idx: dict
        window_size: int
    
    Returns:
        cooccurrence: np.ndarray (V, V)
    """
    V = len(word2idx)
    cooccurrence = np.zeros((V, V), dtype=np.float64)
    
    for sentence in tokenized_corpus:
        indices = [word2idx[w] for w in sentence]
        
        for center_pos in range(len(indices)):
            center_idx = indices[center_pos]
            
            for offset in range(1, window_size + 1):
                # 距離による重み: 1/distance
                weight = 1.0 / offset
                
                # 右側
                if center_pos + offset < len(indices):
                    context_idx = indices[center_pos + offset]
                    cooccurrence[center_idx, context_idx] += weight
                    cooccurrence[context_idx, center_idx] += weight
                
                # 左側
                if center_pos - offset >= 0:
                    # 左側は上で右側として既にカウントされるので不要
                    # （対称行列を構築するため）
                    pass
    
    return cooccurrence


# 共起行列の構築
cooccurrence = build_cooccurrence_matrix(corpus_tokenized, word2idx, window_size=2)

print(f"✅ 共起行列のサイズ: {cooccurrence.shape}")
print(f"   非ゼロ要素数: {np.count_nonzero(cooccurrence)}")
print(f"   スパース度: {1 - np.count_nonzero(cooccurrence)/(vocab_size**2):.2%}")

In [ ]:
# ============================================================
# セクション7.5: 共起行列のヒートマップ
# ============================================================

# 注目単語のサブセットで可視化
focus_words_glove = ['cat', 'dog', 'mouse', 'the', 'on',
                     'rice', 'fish', 'eat', 'drink',
                     'sun', 'moon', 'sky', 'river', 'sea']
focus_indices_glove = [word2idx[w] for w in focus_words_glove]

# サブ行列の抽出
sub_matrix = cooccurrence[np.ix_(focus_indices_glove, focus_indices_glove)]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左: 共起回数（生の値）
sns.heatmap(
    sub_matrix,
    xticklabels=focus_words_glove,
    yticklabels=focus_words_glove,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    square=True,
    ax=axes[0]
)
axes[0].set_title('共起行列（生の共起回数）', fontsize=13)

# 右: log共起回数
log_sub_matrix = np.log1p(sub_matrix)  # log(1 + x) でゼロ対策
sns.heatmap(
    log_sub_matrix,
    xticklabels=focus_words_glove,
    yticklabels=focus_words_glove,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    square=True,
    ax=axes[1]
)
axes[1].set_title('共起行列（log スケール）', fontsize=13)

plt.suptitle('共起行列の可視化 — GloVe はこの行列を因数分解する', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("💡 同じ文脈に出現する単語間で高い共起回数が見られます")
print("   'the' のような機能語はほぼ全ての単語と共起するため値が大きい")
print("   GloVe はこの行列を低ランク近似し、密な埋め込みを得ます")

---

## 8. FastText — サブワードの力

### 8.1 Word2Vec の弱点：未知語問題

Word2Vec と GloVe は、学習データに出現した単語にしかベクトルを割り当てられません。**未知語 (OOV: Out-of-Vocabulary)** には対応できないのです。

例えば、"unhappiness" を学習していなくても、"un-", "happy", "-ness" のような構成要素からベクトルを推定できたら便利です。

### 8.2 FastText のアプローチ：文字 n-gram

**FastText** (Bojanowski et al., 2017) は、単語を **文字 n-gram の集合** として表現します。

例：`"where"` (n=3の場合)

```
1. 特殊文字を付加: "<where>"
2. 文字 3-gram に分解:
   "<wh", "whe", "her", "ere", "re>"
3. 元の単語自体も含む:
   "<where>"
4. 単語のベクトル = 全 n-gram ベクトルの合計
```

### 8.3 FastText の利点

1. **OOV 対応**: 未知語でも n-gram の組み合わせでベクトルを推定
2. **形態素の活用**: 接頭辞・接尾辞・語幹の情報を自然に捉える
3. **低頻度語の改善**: n-gram を共有するため、少数の出現でも情報が豊富
4. **形態素が豊富な言語**: トルコ語、フィンランド語、日本語などで特に効果的

In [ ]:
# ============================================================
# セクション8.4: サブワード分解のデモ
# ============================================================

def extract_char_ngrams(word, min_n=3, max_n=6):
    """
    単語を文字 n-gram に分解する（FastText 方式）。
    
    Parameters:
        word: str - 入力単語
        min_n: int - 最小 n-gram 長
        max_n: int - 最大 n-gram 長
    
    Returns:
        ngrams: list of str - 文字 n-gram のリスト
    """
    # 特殊文字で囲む
    padded = f"<{word}>"
    ngrams = []
    
    for n in range(min_n, min(max_n + 1, len(padded) + 1)):
        for i in range(len(padded) - n + 1):
            ngrams.append(padded[i:i+n])
    
    # 元の単語自体も追加
    ngrams.append(f"<{word}>")
    
    return ngrams


# デモ: いくつかの単語で n-gram を表示
demo_words = ['where', 'unhappiness', 'running', 'cat']

print("📊 FastText 文字 n-gram 分解のデモ:")
print("=" * 60)

for word in demo_words:
    ngrams = extract_char_ngrams(word, min_n=3, max_n=5)
    print(f"\n🔤 '{word}':")
    print(f"   n-gram数: {len(ngrams)}")
    print(f"   n-grams: {ngrams}")

# 共有 n-gram の確認
print("\n" + "=" * 60)
print("\n💡 形態素の共有を確認:")
pairs = [('running', 'runner'), ('unhappy', 'happiness'), ('cat', 'cats')]
for w1, w2 in pairs:
    ng1 = set(extract_char_ngrams(w1, 3, 5))
    ng2 = set(extract_char_ngrams(w2, 3, 5))
    shared = ng1 & ng2
    print(f"   '{w1}' ∩ '{w2}': {len(shared)} 共有 n-gram → {shared}")

In [ ]:
# ============================================================
# セクション8.5: Word2Vec vs GloVe vs FastText 比較表
# ============================================================

# プログラムで比較表を作成し表示
comparison_data = {
    '手法': ['Word2Vec\n(Skip-gram/CBOW)', 'GloVe', 'FastText'],
    '学習方法': [
        'ローカル文脈\n（予測ベース）',
        'グローバル共起\n（カウントベース）',
        'ローカル文脈\n+ サブワード'
    ],
    'OOV対応': ['✗ 不可', '✗ 不可', '✅ 可能'],
    '形態素情報': ['✗ なし', '✗ なし', '✅ あり'],
    '学習速度': ['普通', '速い', 'やや遅い'],
    '必要データ量': ['中程度', '大量', '中程度'],
}

fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off')

# テーブルの作成
columns = list(comparison_data.keys())
cell_text = list(zip(*comparison_data.values()))

table = ax.table(
    cellText=cell_text,
    colLabels=columns,
    cellLoc='center',
    loc='center',
    colColours=['#4472C4'] * len(columns)
)

# スタイリング
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 2.0)

# ヘッダーの文字色を白に
for j in range(len(columns)):
    table[0, j].set_text_props(color='white', fontweight='bold')

# 行ごとに交互に色を付ける
for i in range(len(cell_text)):
    color = '#D9E2F3' if i % 2 == 0 else 'white'
    for j in range(len(columns)):
        table[i + 1, j].set_facecolor(color)

ax.set_title('静的埋め込み手法の比較', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("💡 実用では:")
print("   ・英語のよくある用途 → GloVe の事前学習済みベクトルが便利")
print("   ・多言語・形態素豊富な言語 → FastText が優位")
print("   ・カスタムドメイン → Word2Vec/FastText を自分のコーパスで学習")

---

## 9. 静的埋め込みの限界

### 9.1 多義語問題 (Polysemy)

静的埋め込み（Word2Vec, GloVe, FastText）の最大の弱点は、**1つの単語に1つのベクトル** しか割り当てないことです。

しかし、多くの単語は複数の意味を持ちます：

| 単語 | 意味1 | 意味2 |
|------|-------|-------|
| "bank" | 銀行 | 川岸 |
| "bat" | コウモリ | バット |
| "crane" | 鶴 | クレーン |
| "spring" | 春 | ばね |
| "right" | 正しい | 右 |

静的埋め込みでは "bank" のベクトルは、「銀行」と「川岸」の意味を **混ぜた** ものになります。

In [ ]:
# ============================================================
# セクション9.2: 多義語問題のデモ
# ============================================================

# 多義語を含む拡張コーパスでデモ
polysemy_corpus = [
    # "bank" = 銀行
    "i deposited money at the bank",
    "the bank approved my loan",
    "she works at a large bank",
    "the bank has high interest rates",
    # "bank" = 川岸
    "we sat on the river bank",
    "flowers grow along the bank",
    "the boat docked at the bank",
    "fish swim near the bank",
    # 金融関連
    "money and loan are financial terms",
    "interest rates affect the economy",
    # 自然関連
    "the river flows through the valley",
    "flowers and fish live near water",
]

# トークナイズと語彙構築
poly_tokenized = [s.split() for s in polysemy_corpus]
poly_tokens = [t for s in poly_tokenized for t in s]
poly_counts = Counter(poly_tokens)
poly_vocab = sorted(poly_counts.keys())
poly_word2idx = {w: i for i, w in enumerate(poly_vocab)}
poly_idx2word = {i: w for w, i in poly_word2idx.items()}
poly_V = len(poly_vocab)

# 学習ペアの生成
poly_pairs = generate_training_pairs(poly_tokenized, poly_word2idx, window_size=2)

# 負例分布
poly_neg_probs = build_negative_sampling_distribution(poly_counts, poly_vocab)

# Skip-gram の学習
np.random.seed(42)
poly_model = SkipGramNS(
    vocab_size=poly_V,
    embedding_dim=20,
    neg_probs=poly_neg_probs,
    num_negatives=3
)

# 学習ループ
for epoch in range(600):
    np.random.shuffle(poly_pairs)
    lr = 0.03 * (1.0 - epoch / 600)
    lr = max(lr, 0.0003)
    for c, ctx in poly_pairs:
        poly_model.train_step(c, ctx, lr)

# "bank" の最近傍を確認
print("📊 多義語 'bank' の最近傍:")
print("=" * 50)

neighbors = find_nearest_neighbors('bank', poly_model, poly_word2idx, poly_idx2word, top_k=8)
for rank, (word, sim) in enumerate(neighbors, 1):
    # 意味のラベルを推定
    financial = {'money', 'loan', 'deposited', 'approved', 'interest', 'rates', 'financial', 'works', 'large'}
    nature = {'river', 'flowers', 'fish', 'water', 'boat', 'valley', 'swim', 'grow', 'docked', 'flows'}
    if word in financial:
        label = "💰 金融"
    elif word in nature:
        label = "🌊 自然"
    else:
        label = "   ---"
    print(f"   {rank}. '{word}' (類似度: {sim:.4f}) {label}")

print("\n⚠️ 問題: 'bank' のベクトルは金融と自然の両方の意味を混合している")
print("   → 文脈に応じて異なるベクトルを返す『文脈依存埋め込み』が必要")
print("   → これが Notebook 152 で扱う ELMo, BERT 等の動機")

In [ ]:
# ============================================================
# セクション9.3: 静的 vs 文脈依存埋め込みの概念図
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左: 静的埋め込み ---
ax = axes[0]
# 金融クラスタ
np.random.seed(42)
financial_x = np.random.normal(-1.5, 0.3, 5)
financial_y = np.random.normal(1.5, 0.3, 5)
ax.scatter(financial_x, financial_y, c='blue', s=100, alpha=0.6, label='金融関連')
for i, word in enumerate(['money', 'loan', 'interest', 'deposit', 'financial']):
    ax.annotate(word, (financial_x[i], financial_y[i]), fontsize=8,
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')

# 自然クラスタ
nature_x = np.random.normal(1.5, 0.3, 5)
nature_y = np.random.normal(-1.0, 0.3, 5)
ax.scatter(nature_x, nature_y, c='green', s=100, alpha=0.6, label='自然関連')
for i, word in enumerate(['river', 'flower', 'fish', 'water', 'valley']):
    ax.annotate(word, (nature_x[i], nature_y[i]), fontsize=8,
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')

# "bank" は中間に
bank_x, bank_y = 0.0, 0.3
ax.scatter(bank_x, bank_y, c='red', s=200, marker='*', zorder=5, label='"bank"（混合）')
ax.annotate('bank', (bank_x, bank_y), fontsize=12, fontweight='bold',
            ha='center', va='bottom', xytext=(0, 8), textcoords='offset points', color='red')

# 点線で両クラスタとの関係を示す
ax.plot([bank_x, np.mean(financial_x)], [bank_y, np.mean(financial_y)],
        'r--', alpha=0.3, linewidth=2)
ax.plot([bank_x, np.mean(nature_x)], [bank_y, np.mean(nature_y)],
        'r--', alpha=0.3, linewidth=2)

ax.set_title('静的埋め込み: 1単語 = 1ベクトル', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-3, 3)
ax.set_ylim(-2.5, 3)

# --- 右: 文脈依存埋め込み ---
ax = axes[1]
# 金融クラスタ
ax.scatter(financial_x, financial_y, c='blue', s=100, alpha=0.6, label='金融関連')
for i, word in enumerate(['money', 'loan', 'interest', 'deposit', 'financial']):
    ax.annotate(word, (financial_x[i], financial_y[i]), fontsize=8,
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')

# 自然クラスタ
ax.scatter(nature_x, nature_y, c='green', s=100, alpha=0.6, label='自然関連')
for i, word in enumerate(['river', 'flower', 'fish', 'water', 'valley']):
    ax.annotate(word, (nature_x[i], nature_y[i]), fontsize=8,
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')

# "bank" が文脈に応じて異なる位置に
bank1_x, bank1_y = -1.2, 1.2
bank2_x, bank2_y = 1.2, -0.7
ax.scatter(bank1_x, bank1_y, c='red', s=200, marker='*', zorder=5, label='"bank"（金融文脈）')
ax.annotate('bank\n(金融)', (bank1_x, bank1_y), fontsize=10, fontweight='bold',
            ha='center', va='bottom', xytext=(0, 8), textcoords='offset points', color='red')

ax.scatter(bank2_x, bank2_y, c='orange', s=200, marker='*', zorder=5, label='"bank"（川岸文脈）')
ax.annotate('bank\n(川岸)', (bank2_x, bank2_y), fontsize=10, fontweight='bold',
            ha='center', va='bottom', xytext=(0, 8), textcoords='offset points', color='orange')

ax.set_title('文脈依存埋め込み: 文脈ごとに異なるベクトル', fontsize=13)
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(-3, 3)
ax.set_ylim(-2.5, 3)

plt.suptitle('静的埋め込みの限界と文脈依存埋め込みの動機', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("⚠️ 静的埋め込みでは 'bank' は両方の意味の中間にしか置けない")
print("✅ 文脈依存埋め込みでは、文に応じて異なる位置にマッピングされる")
print("   → 次の Notebook 152 で ELMo, BERT などの手法を学びます")

---

## 10. まとめ・チートシート・よくある間違い・自己評価クイズ

### 10.1 まとめ

このノートブックでは、静的埋め込みの代表的手法を体系的に学びました：

1. **分布仮説**: 「単語はその文脈で決まる」が全手法の共通基盤
2. **Skip-gram**: 中心語から文脈語を予測する予測ベースの手法
3. **負例サンプリング**: ソフトマックスの計算コストを回避する近似手法
4. **CBOW**: 文脈語から中心語を予測（Skip-gram の逆）
5. **GloVe**: 共起行列のlog値をベクトル内積で近似するカウントベースの手法
6. **FastText**: サブワード情報を活用し、OOV語にも対応
7. **限界**: 多義語問題 → 文脈依存埋め込みへの動機

### 10.2 チートシート

| 手法 | 特徴 | 長所 | 短所 |
|------|------|------|------|
| **Word2Vec (SG)** | 中心語→文脈語予測 | 低頻度語に強い、小データOK | OOV不可、多義語未対応 |
| **Word2Vec (CBOW)** | 文脈語→中心語予測 | 高速、頻出語に強い | 低頻度語に弱い |
| **GloVe** | 共起行列の因数分解 | グローバル統計量を活用 | メモリ消費大 |
| **FastText** | サブワード n-gram | OOV対応、形態素活用 | 学習が遅い、モデルサイズ大 |

### 10.3 よくある間違い

1. **❌ 負例サンプリングを使わずにソフトマックスを計算する**
   - 語彙が数万語あると計算が非常に遅くなる
   - 負例サンプリング（K=5～20）で十分な品質が得られる

2. **❌ ウィンドウサイズを大きくしすぎる**
   - 大きすぎると局所的な意味関係が薄まる
   - window=2~5 が一般的。構文関係を捉えたいなら小さく、トピックを捉えたいなら大きく

3. **❌ テキストの前処理を怠る**
   - 小文字化、句読点除去、低頻度語のフィルタリングを忘れると品質が大幅に低下
   - "The" と "the" が別の単語として学習されてしまう

### 10.4 自己評価クイズ

**Q1**: Skip-gram モデルの目的関数を、負例サンプリングを使わない形で書いてください。なぜこれが計算上問題になりますか？

<details>
<summary>回答を表示</summary>

$$P(c \mid w) = \frac{\exp(\mathbf{v}_c' \cdot \mathbf{v}_w)}{\sum_{j=1}^{V} \exp(\mathbf{v}_j' \cdot \mathbf{v}_w)}$$

分母の $\sum_{j=1}^{V}$ が語彙全体にわたる和であるため、毎回 V 個のベクトルとの内積を計算する必要がある。V = 100,000 の場合、1つの学習ペアに対して10万回の内積計算が必要で、これを数十億ペアに対して繰り返すのは非現実的。
</details>

**Q2**: 負例サンプリングの分布で頻度の 0.75 乗を使う理由は？

<details>
<summary>回答を表示</summary>

0.75 乗（3/4 乗）は、低頻度語のサンプリング確率を相対的に引き上げ、高頻度語（"the", "a" など）の支配を緩和する効果がある。1乗（そのままの頻度）だと高頻度語ばかりがサンプリングされ、0乗（一様分布）だと頻度情報が完全に無視される。0.75 はこの間の良いバランスを提供する経験的な値。
</details>

**Q3**: CBOWとSkip-gramのどちらが低頻度語に有利ですか？その理由は？

<details>
<summary>回答を表示</summary>

**Skip-gram** の方が低頻度語に有利。Skip-gram では各中心語から複数の学習ペアが生成されるため、低頻度語でも十分な更新回数を得られる。一方 CBOW では文脈語の平均をとるため、低頻度語の情報が他の文脈語に埋もれやすい。
</details>

**Q4**: GloVe が Word2Vec と異なる最も本質的な点は何ですか？

<details>
<summary>回答を表示</summary>

GloVe は**グローバルな共起統計量**（コーパス全体の共起行列）を直接活用する。Word2Vec はローカルな文脈ウィンドウをスライドさせて個々のペアから学習するのに対し、GloVe は事前に計算した共起行列の対数値をベクトル内積で近似する重み付き最小二乗法を用いる。つまり、Word2Vec は「予測ベース」、GloVe は「カウントベース + 次元圧縮」のアプローチ。
</details>

**Q5**: FastText が "unhappiness" という未知語に対してもベクトルを生成できるのはなぜですか？

<details>
<summary>回答を表示</summary>

FastText は単語を文字 n-gram の集合として表現し、単語ベクトルをそれら n-gram ベクトルの合計として構成する。"unhappiness" が学習データになくても、その構成 n-gram（例: "unh", "nha", "hap", "app", "ppi", "pin", "ine", "nes", "ess" など）の多くは他の単語を通じて学習されている。これらの n-gram ベクトルを合計することで、未知語のベクトルを近似的に推定できる。
</details>

### 10.5 次のステップ

**Notebook 152: Contextual Embeddings（文脈依存埋め込み）** では、以下を扱います：

- ELMo（Embeddings from Language Models）
- BERT（Bidirectional Encoder Representations from Transformers）
- 文脈に応じて変化するベクトル表現
- 多義語問題の解決

---

## 参考文献

1. Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013). *Efficient Estimation of Word Representations in Vector Space*. arXiv:1301.3781
2. Mikolov, T., Sutskever, I., Chen, K., Corrado, G., & Dean, J. (2013). *Distributed Representations of Words and Phrases and their Compositionality*. NIPS 2013
3. Pennington, J., Socher, R., & Manning, C. D. (2014). *GloVe: Global Vectors for Word Representation*. EMNLP 2014
4. Bojanowski, P., Grave, E., Joulin, A., & Mikolov, T. (2017). *Enriching Word Vectors with Subword Information*. TACL 2017
5. Goldberg, Y., & Levy, O. (2014). *word2vec Explained: Deriving Mikolov et al.'s Negative-Sampling Word-Embedding Method*. arXiv:1402.3722